# Dim_Date table - CALENDAR
**Syntax:** `CALENDAR(<start_date>, <end_date>)`

**Example:**

```
Dim_Date = DISTINCT(
  SELECTCOLUMNS(
    CALENDAR(MIN(<Fact_Date>), MAX(<Fact_Date>)),
    "Date", [Date],
    "Year", YEAR([Date]),
    "Quarter", QUARTER([Date])
    "Month", MONTH([Date]),
    "Month Name", FORMAT([Date], "mmm")
  )
)
```

# Filter Function - CALCULATE
**Syntax:** `CALCULATE(<expression>[, <filter1> [, <filter2> [, …]]])`





## Boolean Filter Expressions
An expression evaluates to **_`TRUE` or ` FALSE`_**. Rules:
- They can **_reference columns_** from a single table
- They can **_contain an aggregation function_** that returns a scalar value
- They cannot **_reference measures_**
- They cannot **_use a nested CALCULATE_** function

**Example:**
```
Total sales on the last selected date =
CALCULATE (
    SUM ( Sales[Sales Amount] ),
    'Sales'[OrderDateKey] = MAX ( 'Sales'[OrderDateKey] )
)
```

## Table Filter Expressions - FILTER
It could be a reference to a model table, but more likely it's a function that **_returns a table_** object

**Example:**
```
Total sales Chuck =
CALCULATE (
    SUM('Fact_Orders'[Sales]), //Can be replaced with another measure
    FILTER(Fact_Orders,
        RELATED('Dim_Sales'[Salesperson]) = "Chuck") //RELATED() is used to return values from another table
)
```

![](./images/Datacamp/Filter_Table_Expression_Example.png)

## Filter Modifier Functions
|Function|	Purpose|
|-|-|
|REMOVEFILTERS	|Remove all filters, or filters from one or more columns of a table, or from all columns of a single table.|
|ALL, ALLEXCEPT, ALLNOBLANKROW	|Remove filters from one or more columns, or from all columns of a single table.|
|KEEPFILTERS	|Add filter without removing existing filters on the same columns.|
|USERELATIONSHIP	|Engage an inactive relationship between related columns, in which case the active relationship will automatically become inactive.|
|CROSSFILTER	|Modify filter direction (from both to single, or from single to both) or disable a relationship.|

--> If the **_REMOVEFILTERS_** function is supported by your tool, it's better to use it to remove filters.

**Example:**
```
Total sales ALL =
CALCULATE (
    SUM('Fact_Orders'[Sales]), //Can be replaced with another measure
    ALL(Fact_Orders)
)
```

![](./images/Datacamp/Filter_Modifier_Function_Example.png)

# Counting - COUNT
- `COUNT<column>`: returns the amount of non-blank rows that consist of numbers, dates and strings in column
- `COUNTA<column>`: similar to COUNT(), with addition boolean
- `COUNTBLANKS<column>`: returns the mount of blank rows
- `DISTINCTCOUNT<column>`: returns the amount of distinct values
- `COUNTROWS<table>`: returns the amount of non-blank rows that consist of numbers, dates and strings in table

# Iterating Functions - ...X
- Iterate over each row of a table to perform an expression. It ends with "X"
  - `AVERAGEX(<table>,<expression>)`
  - `COUNTX(<table>,<expression>)`
  - `COUNTAX(<table>,<expression>)`
  - `MAXX(<table>,<expression>,[<variant>])`
  - `MINX(<table>,<expression>,[<variant>])`
  - `SUMX(<table>,<expression>)`
  - `RANKX(<table>,<expression>)`

**Example 1:** SUMX
![](./images/Datacamp/Iterating_Functions_SUMX_Example.png)
- (1), (2): we need to create 2 lines of code for calculated column + measure
- (3): with only **_SUMX_**, we can achieve the same result with 1 line of code and simplify the data model


**Example 2:** RANKX
```
Total Costs RANKX =
    RANKX(
          ALL(Dim_Sales[Region], //Use ALL() to evaluate all rows from the table
          [Total Costs]
    )
```
The result:

![](./images/Datacamp/Iterating_Functions_RANKX_Example.png)

# Logical Functions

## SWITCH
**Syntax:** `SWITCH(<expression>, <value>, <result>[, <value>, <result>]…[, <else>])`
**Purpose:** Evaluates an expression against a list of values and returns one of multiple possible result expressions. This function can be used to avoid having multiple nested IF statements.

**Example:** replace nested IF
```
Performance = SWITCH(TRUE, //Start with TRUE --> when condition = false, go to next expression
[Total_Sales] < 25 000, "Poor",
[Total_Sales] < 50 000, "Below expectations",
[Total_Sales] < 75 000, "Above expectations",
"Exceptional"
)
```
The result:

![](./images/Datacamp/SWITCH_Example_1.png)


**Example:** replace values
```
Performance = SWITCH([Clothing Type]
"Shoes" , 0.15,
"Pants" , 0.20,
"Belts" , 0.30,
"T-shirt" , 0.25,
)
```
The result:

![](./images/Datacamp/SWITCH_Example_2.png)

# Information Functions

## HASONEVALUE
**Syntax:** HASONEVALUE(<columnName>)
**Purpose:** Returns `TRUE` when the context for **_columnName_** has been filtered down to **_one distinct value only_**. Otherwise is `FALSE`

**Example:** we want to get rid of the **_Total_** for **_Actual Transaction Rank_** colum

![](./images/Datacamp/HASONEVALUE_Example_1.png)

1. Adding HASONEVALUE(), we can see it returns FALSE for Total

![](./images/Datacamp/HASONEVALUE_Example_2.png)


2. We use HASONEVALUE() in conjunction with IF in **_Actual Transaction Rank_**
```
Actual Transaction Rank =
IF(HASONEVALUE(Dim_ProductCategory[ProductCategoryName]) = TRUE,
    RANKX(
        ALL('Dim_ProductCategory'[ProductCategoryName]),
        [Actual Transaction Amount]
    ),
  BLANK()
)

```
**The result:**

![](./images/Datacamp/HASONEVALUE_Example_3.png)

# Table Manupulation Functions

## ADDCOLUMNS
- **Syntax:** `ADDCOLUMNS(<table>, <name>, <expression>[, <name>, <expression>]…)`
- **Purpose:** Adds calculated columns to the given table or table expression.

**Example:**
```
ADDCOLUMNS(ProductCategory
               , "Internet Sales", SUMX(RELATEDTABLE(InternetSales_USD), InternetSales_USD[SalesAmount_USD])
               , "Reseller Sales", SUMX(RELATEDTABLE(ResellerSales_USD), ResellerSales_USD[SalesAmount_USD]))
```

**The result:**

![](./images/Microsoft/ADDCOLUMNS_Example.png)

## DISTINCT
- **Syntax:** `DISTINCT(<table>)`
- **Purpose:** Returns a table by removing duplicate rows from another table or expression.

## SELECTCOLUMNS
- **Syntax:** `SELECTCOLUMNS(<Table>, [<Name>], <Expression>, [<Name>], …) `
- **Purpose:** Returns a table with selected columns from the table and new columns specified by the DAX expressions.

**Example:**
```
SELECTCOLUMNS(Customer, "Country, State", [Country]&", "&[State])
```

**The result:**

![](./images/Microsoft/SELECTCOLUMNS_Example.png)

## SUMMARIZE
- **Syntax:** `SUMMARIZE (<table>, <groupBy_columnName>[, <groupBy_columnName>]…[, <name>, <expression>]…)`
- **Purpose:** Returns a summary table for the requested totals over a set of groups.

**Example:**
```
SUMMARIZE(ResellerSales_USD
      , DateTime[CalendarYear]
      , ProductCategory[ProductCategoryName]
      , "Sales Amount (USD)", SUM(ResellerSales_USD[SalesAmount_USD])
      , "Discount Amount (USD)", SUM(ResellerSales_USD[DiscountAmount])
      )
```

**The result:**

![](./images/Microsoft/SUMMARIZE_Example.png)

### Best Practices
Although it is possible to cerate new columns with **_`SUMMARIZE()`_**, best practice is to wrap **_`ADDCOLUMNS()`_** around **_`SUMMARIZE()`_** when **_creating new columns_**
  - **_`ADDCOLUMNS()`_** will always use row context and will produce the same but consistent result compared to **_`SUMMARIZE()`_**


![](./images/Datacamp/SUMMARIZE_Best_Pratice_Example.png)

# Time Intelligence Function

## Important: Marking Date Table
PBI needs to know data table to create date hierarchies which assist to create correct visuals and calculateions

![](./images/Datacamp/Makring_Date_Table_Step_1.png)
![](./images/Datacamp/Makring_Date_Table_Step_2.png)

## DATESBETWEEN
- **Syntax:** `DATESBETWEEN(<Dates>, <StartDate>, <EndDate>)`
- **Purpose:** Returns a **_table_** that contains a column of dates that begins with a specified start date and continues until a specified end date.

**Example:**
``` 
Sales[Avg Sales Last 30D] =
VAR Last30D =
    DATESBETWEEN ( 
        'Date'[Date], 
        MAX ( 'Date'[Date] ) - 29,  -- boundaries are included, this is why we use 29
        MAX ( 'Date'[Date] )        -- to obtain 30 days
    )
VAR Result =
    CALCULATE (
        [Sales Amount] / 30,
        Last30D
    )
RETURN
    Result
```

**The result:**

![](./images/Datacamp/DATESBETWEEN_Example.png)

## NEXTDAY
- **Syntax:** `NEXTDAY(<dates>)`
- **Purpose:** Returns a **_table_** that contains a **_column of all dates_** from the next day, based on the first date specified in the dates column in the current context.

**Example:**
```
Next Date = NEXTDAY(Dim_Invoice[Date])
```
![](./images/Datacamp/NEXTDAY_Example.png)

## SAMEPERIODLASTYEAR
- **Syntax:** `SAMEPERIODLASTYEAR(<dates>)`
- **Purpose:** Returns a table that contains a column of dates shifted one year back in time from the dates in the specified dates column, in the current context.

**Example:**
```
Measure YoY % =
  VAR __PriorYear = CALCULATE(<Measure>, SAMEPERIODLASTYEAR(<'Date Table'[Date]>))
  RETURN
    DIVIDE(<Measure> - __PriorYear, __PriorYear)
```

**Alternative:**
```
Measure YoY % =
  VAR __PriorYear = CALCULATE(<Measure>, DATEADD(<'Date Table'[Date].[Date]>, -1, YEAR))
  RETURN
    DIVIDE(<Measure> - __PriorYear, __PriorYear)
```

## TOTALMTD, TOTAL QTD, TOTALYTD
- **Syntax:** 
  - `TOTALMTD(<expression>,<dates>[,<filter>])`
  - `TOTALQTD(<expression>,<dates>[,<filter>])`
  - `TOTALYTD(<expression>,<dates>[,<filter>][,<year_end_date>])`
- **Purpose:** Evaluates the value of the expression for the month to date / quarter to date / year to date, in the current context.

**Example:**
```
TOTALYTD([Total Sales], Dim_InvoiceDate[Date])
```

![](./images/Datacamp/TOTALYTD_Example.png)

# Row Level Security

**Example:** Requiremens

![](./images/Datacamp/RLS_Example.png)
- Filter data:
  - (1) 1st condition: use **_Email_** with **USEPRINCIPALNAME()** to filter data associated with **_Dim_Employee_** dimension
  - (2) 2nd condition: filter only if **_Employee_** is marked as `Is_Salesperson = TRUE`
- Calculate data from 1 dimension to another dimension
  - (3), (4) 2 dimensions have **_single direction_** to fact table, but we want to calculate number of city to a specific Employee

**The view of no RLS (all data):**

![](./images/Datacamp/RLS_Example_1.png)

## Manage Security Roles - USERPRINCIPALNAME
**Setup:**

![](./images/Datacamp/RLS_Example_2.png)

**The result:** data from Fact table is correct but calculated data from another dimension is not, due to **_single direction_**

![](./images/Datacamp/RLS_Example_3.png)

## Change Bi-Direction Between Dimension Tables
**Setup:**

![](./images/Datacamp/RLS_Example_4.png)

**The result:**

![](./images/Datacamp/RLS_Example_5.png)